<a href="https://colab.research.google.com/github/Evgeniya-Ryasnova/generative-mocap/blob/master/generative_mocap_reproduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generative MoCap Reproduction Study

## 0. Reproduction Notes

### Goal

* Retrain cGAN models.
* Reproduce the results from the original paper.

### Environment

* Google Colab
* NVIDIA Tesla T4
* Python 3.12
* PyTorch 2.x

### Dataset

* data/data_1.pickle
* data/data_2.pickle

# 1. Environment Setup

Verify the Python, PyTorch and CUDA configuration.

In [3]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
Tesla T4


# 2. Clone Repository

Clone the forked repository and access project files.

In [4]:
!git clone https://github.com/Evgeniya-Ryasnova/generative-mocap.git


Cloning into 'generative-mocap'...
remote: Enumerating objects: 243, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 243 (delta 9), reused 0 (delta 0), pack-reused 220 (from 4)
Receiving objects: 100% (243/243), 1.38 GiB | 18.89 MiB/s, done.
Resolving deltas: 100% (30/30), done.
Updating files: 100% (184/184), done.


# 3. Change Working Directory

The working directory is changed to the cloned repository so that Python can access the project scripts, datasets, and output folders correctly.

In [5]:
%cd generative-mocap

/content/generative-mocap


# 4. Install Dependencies

Install required packages.

In [6]:
!pip install spm1d ezc3d statsmodels

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 62.6 MB/s eta 0:00:00


# 5. Verify Training Function

The main training function `train_cgan()` is imported and its function signature is inspected to understand which parameters can be changed for reproduction experiments.

In [7]:
from train_cgans import train_cgan
import inspect

print(inspect.signature(train_cgan))

/content/generative-mocap/utils.py:486: SyntaxWarning: invalid escape sequence '\%'
  for i in [4, 5, 9]: axs[i].set_xlabel('Gait Cycle [\%]', fontsize=25)
/content/generative-mocap/utils.py:598: SyntaxWarning: invalid escape sequence '\%'
  for ax in axs[2:]: ax.set_xlabel('Gait Cycle [\%]', fontsize=25)


(marker_names=array(['L_IAS', 'L_IPS', 'R_IPS', 'R_IAS', 'R_FTC', 'R_FLE', 'R_FME',
       'R_FAX', 'R_TTC', 'R_FAL', 'R_TAM', 'R_FCC', 'R_FM1', 'R_FM2',
       'R_FM5'], dtype='<U5'), grf_names=array(['force', 'point', 'moment'], dtype='<U6'), ik_names=array(['pelvis_tilt', 'pelvis_list', 'pelvis_rotation', 'hip_flexion_r',
       'hip_adduction_r', 'hip_rotation_r', 'knee_angle_r',
       'ankle_angle_r'], dtype='<U15'), value_cols=['marker_gc', 'grf_3d_gc', 'ik_gc'], names_cols=['marker_names', 'grf_names_3d', 'ik_names'], label_col_contd=['age'], label_col_discr=None, data_df_file=['data/data_1.pickle', 'data/data_2.pickle'], excluded_subjects=[2014001, 2014003, 2015042], z_dim=20, hidden_dim=512, n_epochs=3000, lr=0.002, batch_size=128, display_step=250, n_samples=10, label_contd_lims=[[15, 71, 1]], label_discr_lims=None, model_name='acgan', device=device(type='cuda'), seed=0)


# 6. Test WS-cGAN Training

A short WS-cGAN training run is performed with 10 epochs to verify that the full training pipeline works before running the full 3000-epoch experiment.

## Parameters

- n_epochs = 10
- display_step = 1
- model_name = "wscgan_test"
- condition = walking_speed

In [8]:
from train_cgans import train_cgan
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_cgan(
    label_col_contd=['walking_speed'],
    label_col_discr=None,
    n_epochs=10,
    display_step=1,
    label_contd_lims=[[0.16, 2.41, 0.02]],
    label_discr_lims=None,
    model_name='wscgan_test',
    device=device,
    seed=0
)

/content/generative-mocap/train_cgans.py:390: UserWarning: The torch.cuda.*DtypeTensor constructors are no longer recommended. It's best to use methods such as torch.tensor(data, dtype=*, device='cuda') to create tensors. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:78.)
  valid = Tensor(real.shape[0], 1).fill_(1.0)


Epoch 1, step 1: Generator loss: 0.06170816719532013, discriminator loss: 0.6723203659057617
Epoch 1, step 2: Generator loss: 0.04478055238723755, discriminator loss: 0.6367135047912598
Epoch 1, step 3: Generator loss: 0.04045671224594116, discriminator loss: 0.650017499923706
Epoch 1, step 4: Generator loss: 0.02108882926404476, discriminator loss: 0.6738141775131226
Epoch 1, step 5: Generator loss: 0.016384847462177277, discriminator loss: 0.6306586265563965
Epoch 1, step 6: Generator loss: 0.012760194949805737, discriminator loss: 0.5233613848686218
Epoch 2, step 7: Generator loss: 0.009743797592818737, discriminator loss: 0.4933582544326782
Epoch 2, step 8: Generator loss: 0.00881848856806755, discriminator loss: 0.47737598419189453
Epoch 2, step 9: Generator loss: 0.00764224398881197, discriminator loss: 0.4228903353214264
Epoch 2, step 10: Generator loss: 0.006682139355689287, discriminator loss: 0.4684189558029175
Epoch 2, step 11: Generator loss: 0.00585491769015789, discrimina

# 7. Check Training Outputs

The output directory is inspected to confirm that the trained generator, synthetic data, and transformation files were saved successfully.

In [9]:
!ls Results/wscgan_test

back_transform_input.pkl  synthetic_data_after_training.pkl
generator.pt		  transform_labels.pkl


# 8. Notes

### Observations

* The original examples in train_cgans.py are enclosed in triple quotes and are therefore not executed automatically.
* Python 3.12 raises a SyntaxWarning related to '%', but training is unaffected.
* PyTorch 2.x raises deprecation warnings regarding torch.cuda.*Tensor constructors.
* Training completed successfully on an NVIDIA Tesla T4 GPU.

### Current Status

✅ Repository cloned successfully

✅ Dependencies installed successfully

✅ Data loaded successfully

✅ GPU available and functioning

✅ Test WS-cGAN training completed successfully

✅ Synthetic data generated successfully